# 201. Agent 并行工具调用：call id、乱序结果与幂等怎样实现？

> **面试问题：多个 tool call 并行执行、结果乱序返回时，怎样通过 call id、状态机、顺序汇合和 result ledger 保证 Agent loop 一致？**

## 先给结论

面试中不能只背术语；需要把数学坐标、消息状态、协议顺序或模板字节流变成可检验的状态机。下列代码只使用标准库和小数组，明确教学 oracle 与生产替换点；它们不等同于真实模型效果、网络可靠性或正式安全认证。

## 一手资料

- [Toolformer](https://arxiv.org/abs/2302.04761)
- [ReAct](https://arxiv.org/abs/2210.03629)
- [MCP Tools](https://modelcontextprotocol.io/specification/draft/server/tools)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "explicit-assertions", "production": "versioned-and-observed"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "explicit-assertions"  # 执行本行的状态、计算或校验逻辑。
assert "observed" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：并行 tool call 的一致性靠 call id，而非自然语言顺序

模型一次输出多个工具调用时，结果可能乱序返回。每个 call 应有唯一 id、工具名、参数摘要、状态和幂等键；tool result 必须精确指向 pending call，不能按“第几个结果”或文本相似度匹配。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass  # 执行本行的状态、计算或校验逻辑。
class ToolCall:  # 执行本行的状态、计算或校验逻辑。
    call_id: str  # 执行本行的状态、计算或校验逻辑。
    name: str  # 执行本行的状态、计算或校验逻辑。
    arguments: dict  # 执行本行的状态、计算或校验逻辑。
    status: str = "pending"  # 执行本行的状态、计算或校验逻辑。
calls = [ToolCall("c-1", "weather", {"city": "上海"}), ToolCall("c-2", "calendar", {"date": "2026-08-01"})]  # 执行本行的状态、计算或校验逻辑。
assert {call.call_id for call in calls} == {"c-1", "c-2"}  # 执行本行的状态、计算或校验逻辑。
assert all(call.status == "pending" for call in calls)  # 执行本行的状态、计算或校验逻辑。
assert calls[0].name == "weather"  # 执行本行的状态、计算或校验逻辑。


## 2. 注册表：call id、工具名和参数 schema 都应在执行前冻结

工具执行器只接收经过验证的 call。示例检查 id 唯一、工具 allow-list 和必需参数；生产还应做 JSON Schema、主体权限、速率限制、超时与参数 canonicalization。


In [ ]:
tool_requirements = {"weather": {"city"}, "calendar": {"date"}}  # 执行本行的状态、计算或校验逻辑。
def validate_calls(calls):  # 执行本行的状态、计算或校验逻辑。
    ids = [call.call_id for call in calls]  # 执行本行的状态、计算或校验逻辑。
    return len(ids) == len(set(ids)) and all(call.name in tool_requirements and tool_requirements[call.name].issubset(call.arguments) for call in calls)  # 执行本行的状态、计算或校验逻辑。
assert validate_calls(calls)  # 执行本行的状态、计算或校验逻辑。
assert not validate_calls([calls[0], ToolCall("c-1", "calendar", {"date": "x"})])  # 执行本行的状态、计算或校验逻辑。
assert not validate_calls([ToolCall("c-3", "weather", {})])  # 执行本行的状态、计算或校验逻辑。


## 3. 分派：并行不等于失去 trace 顺序

可并行的调用在执行器中可以同时启动，但 trace 必须记录原始计划顺序、开始时间和幂等 key。这里返回稳定的 dispatch envelope，避免依赖 Python 对象地址或模型输出顺序。


In [ ]:
def dispatch(calls, trace_id):  # 执行本行的状态、计算或校验逻辑。
    return [{"trace": trace_id, "ordinal": index, "call_id": call.call_id, "tool": call.name} for index, call in enumerate(calls)]  # 执行本行的状态、计算或校验逻辑。
envelopes = dispatch(calls, "t-1")  # 执行本行的状态、计算或校验逻辑。
assert [item["ordinal"] for item in envelopes] == [0, 1]  # 执行本行的状态、计算或校验逻辑。
assert envelopes[0]["call_id"] == "c-1"  # 执行本行的状态、计算或校验逻辑。
assert all(item["trace"] == "t-1" for item in envelopes)  # 执行本行的状态、计算或校验逻辑。


## 4. 结果提交：只接受对应 pending call 的一次结果

结果可以乱序，但提交时需要按 call id 找到 pending 状态。未知 id、重复 result、调用失败和 schema 不符必须显式留下失败事件；不要让后到结果悄悄覆盖先到结果。


In [ ]:
def commit_result(calls, call_id, result):  # 执行本行的状态、计算或校验逻辑。
    lookup = {call.call_id: call for call in calls}  # 执行本行的状态、计算或校验逻辑。
    call = lookup.get(call_id)  # 执行本行的状态、计算或校验逻辑。
    if call is None or call.status != "pending":  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("结果不对应唯一的 pending call")  # 执行本行的状态、计算或校验逻辑。
    call.status = "completed"  # 执行本行的状态、计算或校验逻辑。
    return {"call_id": call_id, "result": result}  # 执行本行的状态、计算或校验逻辑。
late_first = commit_result(calls, "c-2", {"slots": 3})  # 执行本行的状态、计算或校验逻辑。
assert late_first["call_id"] == "c-2"  # 执行本行的状态、计算或校验逻辑。
assert calls[1].status == "completed"  # 执行本行的状态、计算或校验逻辑。
assert calls[0].status == "pending"  # 执行本行的状态、计算或校验逻辑。


## 5. 汇合：向模型发送结果时恢复计划顺序并携带 id

即使 calendar 先完成，下一轮模型上下文仍应按原始 call 顺序组织结果，并显式带上 `tool_call_id`。这让模型、日志和评测器都能追踪哪个 observation 对应哪个 action。


In [ ]:
second = commit_result(calls, "c-1", {"temperature_c": 28})  # 执行本行的状态、计算或校验逻辑。
results = {late_first["call_id"]: late_first["result"], second["call_id"]: second["result"]}  # 执行本行的状态、计算或校验逻辑。
ordered_messages = [{"tool_call_id": call.call_id, "content": results[call.call_id]} for call in calls]  # 执行本行的状态、计算或校验逻辑。
assert [item["tool_call_id"] for item in ordered_messages] == ["c-1", "c-2"]  # 执行本行的状态、计算或校验逻辑。
assert ordered_messages[0]["content"]["temperature_c"] == 28  # 执行本行的状态、计算或校验逻辑。
assert all(call.status == "completed" for call in calls)  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：重复、未知和部分失败不可静默吞掉

部分成功不是全成功。Agent loop 应把每个 tool result 的可重试性、错误码和已产生的副作用分开处理；对同一 call id 的重复提交直接拒绝，避免重复写入和错误计分。


In [ ]:
try:  # 执行本行的状态、计算或校验逻辑。
    commit_result(calls, "c-1", {"temperature_c": 29})  # 执行本行的状态、计算或校验逻辑。
    assert False  # 执行本行的状态、计算或校验逻辑。
except ValueError:  # 执行本行的状态、计算或校验逻辑。
    assert calls[0].status == "completed"  # 执行本行的状态、计算或校验逻辑。
try:  # 执行本行的状态、计算或校验逻辑。
    commit_result(calls, "unknown", {})  # 执行本行的状态、计算或校验逻辑。
    assert False  # 执行本行的状态、计算或校验逻辑。
except ValueError:  # 执行本行的状态、计算或校验逻辑。
    assert len(results) == 2  # 执行本行的状态、计算或校验逻辑。


## 7. 幂等：重试要复用 call id 与已保存的结果

网络重试时不能生成一个看似相同但新的 call id，否则工具端无法去重。这里展示 result ledger 的读优先路径；生产 ledger 应持久化、受租户隔离并关联实际副作用回执。


In [ ]:
result_ledger = {item["tool_call_id"]: item["content"] for item in ordered_messages}  # 执行本行的状态、计算或校验逻辑。
def retry_or_read(call_id, ledger):  # 执行本行的状态、计算或校验逻辑。
    return {"replayed": call_id in ledger, "result": ledger.get(call_id)}  # 执行本行的状态、计算或校验逻辑。
retry = retry_or_read("c-1", result_ledger)  # 执行本行的状态、计算或校验逻辑。
assert retry["replayed"] is True  # 执行本行的状态、计算或校验逻辑。
assert retry["result"]["temperature_c"] == 28  # 执行本行的状态、计算或校验逻辑。
assert retry_or_read("c-9", result_ledger)["result"] is None  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：把计划、调用、结果和 schema 版本串起来

排障必须能区分模型选择错误、参数错误、工具错误和 join 错误。因此 trace 制品至少保存 call id、tool version、参数摘要、开始/结束状态、结果摘要和模型/template 版本。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"trace": "t-1", "calls": [call.call_id for call in calls], "tools": [call.name for call in calls], "joined": [item["tool_call_id"] for item in ordered_messages]}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["calls"] == artifact["joined"]  # 执行本行的状态、计算或校验逻辑。
assert len(artifact["tools"]) == 2  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时先说明不变量，再给出主路径和失败分支，最后说明指标、版本制品与生产替换点。不要把一个受控样例的通过误报成模型质量、可靠网络或端到端安全保证。
